# Deliverable 4: Dynamics-Loss-Gated VLM Intervention — DreamerV3 + Gemini on Boxing

**Course:** Deep Learning Project  
**Authors:** Iqra Khurram (27100376) | Xeerak Azhar (27100310)

---

## What this notebook does

We run DreamerV3 (Hafner et al., 2023/2025) with a **VLM surprise gate** on **Atari Boxing**
using the Atari100k config (400K environment steps).

### Why Boxing instead of Montezuma's Revenge?

| Factor | Montezuma's Revenge | Boxing |
|--------|-------------------|--------|
| Paper score (400K) | ~0–100 | **82** (gamer median) |
| Paper score (200M) | 1,852 | **100** (max possible) |
| Reward type | Extremely sparse | **Dense** (points per hit) |
| Exploration | Hardest in Atari | Straightforward |
| Reproducibility | Known ALE issues | **Confirmed reproducible** |

Boxing is ideal because:
1. **Dense rewards** → the agent gets feedback every few frames
2. **DreamerV3 scores 100** at 200M steps (perfect score) and **82 at 400K** in paper
3. **Confirmed reproducible** with current ale-py 0.9.0 (GitHub issue #175)
4. The VLM gate can meaningfully assist with tactical decisions

### Architecture
- **DreamerV3** handles all learning: CNN encoder → RSSM world model → actor-critic
- **VLM surprise gate** monitors pixel-level frame differences as a proxy for dynamics loss
- When surprise spikes (z-score > threshold), **Gemini 1.5 Flash** provides action advice
- VLM reward bonus is shaped and added to environment reward

**Citation:** Hafner, D., Pasukonis, J., Ba, J., & Lillicrap, T. (2025). Mastering diverse control tasks through world models. *Nature*. Code: https://github.com/danijar/dreamerv3

---
## Cell 1: JAX Reinstall (ONLY if needed)

**Uncomment and run ONLY if you get a JAX version error.**  
After running, **restart kernel**, skip this cell, and run all remaining cells.

In [3]:
# UNCOMMENT BELOW ONLY IF JAX VERSION ERRORS -- then restart kernel, skip this cell

# import sys, subprocess
# freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()
# targets = sorted({
#     line.split("==")[0].split("[")[0].strip().lower()
#     for line in freeze
#     if line.split("==")[0].split("[")[0].strip().lower().startswith(("jax", "jaxlib"))
# })
# if targets:
#     subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", *targets], check=False)
# subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade",
#                 "jax[cuda12]==0.4.33", "numpy<2"], check=True)
# print("RESTART KERNEL NOW, then skip this cell and run the rest.")

## Cell 2: Verify Environment

In [4]:
import os, jax, jaxlib
import numpy as np

print("jax    :", jax.__version__)
print("jaxlib :", jaxlib.__version__)
print("numpy  :", np.__version__)
print("devices:", jax.devices())

# Sanity checks
assert jax.devices(), "No GPU/TPU found -- enable GPU in Kaggle settings!"
print("\nEnvironment OK.")

jax    : 0.7.2
jaxlib : 0.7.2
numpy  : 2.0.2
devices: [CudaDevice(id=0), CudaDevice(id=1)]

Environment OK.


## Cell 3: Clone DreamerV3 + Install Dependencies

In [5]:
import os, subprocess, sys

REPO_DIR = "/kaggle/working/dreamerv3"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/danijar/dreamerv3.git", REPO_DIR], check=True)
    print("Cloned dreamerv3 repo.")
else:
    print("Repo already exists.")

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

# Install pinned dependencies (ale-py==0.9.0 is critical for reproducibility)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-r", "requirements.txt"],
    check=True,
)
print("\nDependencies installed.")

Cloning into '/kaggle/working/dreamerv3'...


Cloned dreamerv3 repo.
CWD: /kaggle/working/dreamerv3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 183.0 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of optax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires ale-py>=0.10.1, but you have ale-py 0.9.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9",


Dependencies installed.


In [6]:
# Install Gemini SDK for VLM gate
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q",
     "google-generativeai", "Pillow"],
    check=True,
)
print("Gemini SDK installed.")

Gemini SDK installed.


## Cell 5: Load Gemini API Key

Add your key as a Kaggle Secret named `GEMINI_API_KEY`.  
Free tier from [Google AI Studio](https://aistudio.google.com/app/apikey): 15 req/min, 1500 req/day.

In [7]:
import os

# Try Kaggle secrets first, then env variable, then empty (VLM disabled)
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("API key loaded from Kaggle secrets.")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    if GEMINI_API_KEY:
        print("API key loaded from environment.")
    else:
        print("WARNING: No Gemini API key found. VLM calls will be skipped.")
        print("Training will still run (DreamerV3 baseline only).")

Training will still run (DreamerV3 baseline only).


## Cell 6: Write VLM Gate Module

The surprise gate monitors frame-to-frame pixel MSE. When it spikes beyond
a z-score threshold, it calls Gemini for action advice + reward shaping.

**Key fix from previous notebook:** The VLM prompt is now adapted for Boxing
(not Montezuma's Revenge), and reward keywords match Boxing gameplay.

In [8]:
VLM_GATE_SRC = r'''
import json
import numpy as np
from collections import deque

try:
    import google.generativeai as genai
    _GENAI_AVAILABLE = True
except Exception:
    _GENAI_AVAILABLE = False

# Boxing uses 18 discrete actions but only a few matter:
# 0=NOOP, 1=FIRE, 2=UP, 3=RIGHT, 4=LEFT, 5=DOWN,
# 6=UPRIGHT, 7=UPLEFT, 8=DOWNRIGHT, 9=DOWNLEFT,
# 10=UPFIRE, 11=RIGHTFIRE, 12=LEFTFIRE, 13=DOWNFIRE,
# 14=UPRIGHTFIRE, 15=UPLEFTFIRE, 16=DOWNRIGHTFIRE, 17=DOWNLEFTFIRE
ACTION_NAMES = {
    0: "NOOP", 1: "FIRE", 2: "UP", 3: "RIGHT", 4: "LEFT", 5: "DOWN",
    6: "UPRIGHT", 7: "UPLEFT", 8: "DOWNRIGHT", 9: "DOWNLEFT", 10: "UPFIRE",
    11: "RIGHTFIRE", 12: "LEFTFIRE", 13: "DOWNFIRE", 14: "UPRIGHTFIRE",
    15: "UPLEFTFIRE", 16: "DOWNRIGHTFIRE", 17: "DOWNLEFTFIRE",
}
NAME_TO_ACTION = {v: k for k, v in ACTION_NAMES.items()}

# Boxing-specific reward keywords
SITUATION_REWARD = {
    "punch": 0.3, "hit": 0.3, "jab": 0.25, "attack": 0.2,
    "close": 0.15, "approach": 0.1, "corner": 0.2,
    "winning": 0.3, "leading": 0.2, "scoring": 0.25,
    "dodge": 0.1, "block": 0.1, "retreat": -0.05,
    "losing": -0.15, "behind": -0.1, "missed": -0.05,
    "far": -0.1, "stuck": -0.2, "idle": -0.15,
}


def _parse_reward_from_description(description):
    desc_lower = description.lower()
    bonus = 0.0
    for keyword, value in SITUATION_REWARD.items():
        if keyword in desc_lower:
            bonus += value
    return float(np.clip(bonus, -1.0, 1.0))


class VLMGate:
    """Surprise-gated VLM advisor for Atari Boxing.

    Monitors pixel MSE between frames. On surprise spikes,
    calls Gemini for tactical advice and queues override actions
    with decaying reward bonus.
    """

    def __init__(self, api_key="", threshold_z=2.5, queue_len=6,
                 cooldown_steps=80, reward_scale=0.25, warmup_steps=5000,
                 enabled=True, log_path="/kaggle/working/vlm_call_log.json"):
        self.threshold_z    = threshold_z
        self.queue_len      = queue_len
        self.cooldown_steps = cooldown_steps
        self.reward_scale   = reward_scale
        self.warmup_steps   = warmup_steps
        self.enabled        = enabled
        self.log_path       = log_path

        self._queue        = deque()
        self._reward_queue = deque()
        self._prev_frame   = None
        self._surprise_buf = deque(maxlen=150)
        self._cooldown     = 0

        self.total_vlm_calls = 0
        self.total_steps     = 0
        self._call_log       = []

        self._model = None
        if enabled and api_key and _GENAI_AVAILABLE:
            genai.configure(api_key=api_key)
            self._model = genai.GenerativeModel("gemini-1.5-flash")
            print(f"[VLMGate] Gemini ready  threshold_z={threshold_z}  "
                  f"queue_len={queue_len}  cooldown={cooldown_steps}  "
                  f"warmup={warmup_steps}  reward_scale={reward_scale}")
        elif not api_key:
            print("[VLMGate] No API key -- VLM calls will be skipped.")
        elif not _GENAI_AVAILABLE:
            print("[VLMGate] google-generativeai not installed.")

    def reset_episode(self):
        self._queue.clear()
        self._reward_queue.clear()
        self._prev_frame = None
        self._cooldown = 0

    def _surprise(self, frame):
        frame_f = frame.astype(np.float32)
        if self._prev_frame is None:
            self._prev_frame = frame_f
            return 0.0
        mse = float(np.mean((frame_f - self._prev_frame) ** 2))
        self._prev_frame = frame_f
        return mse

    def _should_trigger(self, surprise):
        self._surprise_buf.append(surprise)
        if len(self._surprise_buf) < 20:
            return False
        arr = np.array(self._surprise_buf, dtype=np.float32)
        z = (surprise - float(arr.mean())) / (float(arr.std()) + 1e-6)
        return bool(z > self.threshold_z)

    def _call_vlm(self, frame, step, score, lives, action_history):
        if self._model is None:
            return [], 0.0, "no-model"

        from PIL import Image
        import io

        try:
            frame_u8 = np.clip(frame, 0, 255).astype(np.uint8)
            if frame_u8.ndim == 2:
                pil = Image.fromarray(frame_u8, "L").convert("RGB")
            elif frame_u8.shape[-1] == 1:
                pil = Image.fromarray(frame_u8[:, :, 0], "L").convert("RGB")
            else:
                pil = Image.fromarray(frame_u8)

            hist_names = [ACTION_NAMES.get(int(a), "NOOP") for a in action_history[-6:]]

            prompt = (
                "You are helping an AI agent play Atari Boxing.\n"
                "The game is a top-down boxing match. The player (white) "
                "scores by punching the opponent (black). Each hit = +1 point.\n"
                "Return ONLY a JSON object -- no markdown, no backticks.\n\n"
                f"Current state: step={step}  score={score:.0f}\n"
                f"Recent actions: {hist_names}\n\n"
                "Required JSON fields:\n"
                '  "description": 1-2 sentences about the game state '
                '(player position, opponent position, who is winning).\n'
                f'  "actions": list of {self.queue_len} action names from '
                '  [NOOP, FIRE, UP, RIGHT, LEFT, DOWN, UPFIRE, RIGHTFIRE, '
                '  LEFTFIRE, DOWNFIRE].\n'
                '  Prefer FIRE/RIGHTFIRE/LEFTFIRE to punch. '
                '  Move toward opponent then punch.\n'
            )

            resp = self._model.generate_content([prompt, pil])
            text = resp.text.strip()

            # Clean markdown fences if present
            if text.startswith("```"):
                text = text.split("\n", 1)[-1]
            if text.endswith("```"):
                text = text.rsplit("```", 1)[0]
            text = text.strip()

            data = json.loads(text)
            desc    = str(data.get("description", ""))
            raw_act = data.get("actions", [])

            actions = []
            for a in raw_act[:self.queue_len]:
                name = str(a).upper().strip()
                if name in NAME_TO_ACTION:
                    actions.append(NAME_TO_ACTION[name])
                else:
                    actions.append(1)  # Default to FIRE

            # Pad if VLM returned fewer actions
            while len(actions) < self.queue_len:
                actions.append(1)  # FIRE

            bonus = _parse_reward_from_description(desc) * self.reward_scale
            return actions, bonus, desc

        except Exception as e:
            print(f"[VLMGate] VLM call failed: {e}")
            return [], 0.0, f"error: {e}"

    def get_action(self, frame, agent_action, step, score, lives, action_history):
        self.total_steps += 1

        # If we have queued VLM actions, use them
        if self._queue:
            override_action = self._queue.popleft()
            bonus = self._reward_queue.popleft() if self._reward_queue else 0.0
            return override_action, True, bonus

        # Compute surprise
        surprise = self._surprise(frame)

        # Decrement cooldown
        if self._cooldown > 0:
            self._cooldown -= 1
            return agent_action, False, 0.0

        # Skip during warmup
        if step < self.warmup_steps:
            return agent_action, False, 0.0

        # Check if surprise triggers VLM
        if self._should_trigger(surprise):
            actions, bonus, desc = self._call_vlm(
                frame, step, score, lives, action_history
            )
            if actions:
                self.total_vlm_calls += 1
                self._cooldown = self.cooldown_steps

                # Queue actions with linearly decaying reward bonus
                self._queue.extend(actions)
                for i, _ in enumerate(actions):
                    decay = 1.0 - (i / len(actions))
                    self._reward_queue.append(bonus * decay)

                self._call_log.append({
                    "step": step, "surprise": float(surprise),
                    "actions": [ACTION_NAMES.get(a, "?") for a in actions],
                    "reward_bonus": float(bonus),
                    "description": desc[:200],
                })

                print(f"[VLMGate] step={step:>7d}  surprise={surprise:.1f}  "
                      f"bonus={bonus:+.3f}  actions={[ACTION_NAMES.get(a,'?') for a in actions[:3]]}...")

                # Return the first queued action
                return self._queue.popleft(), True, self._reward_queue.popleft()

        return agent_action, False, 0.0

    def save_log(self):
        log = {
            "total_steps": self.total_steps,
            "total_vlm_calls": self.total_vlm_calls,
            "call_pct": (self.total_vlm_calls / max(1, self.total_steps)) * 100,
            "calls": self._call_log,
        }
        with open(self.log_path, "w") as f:
            json.dump(log, f, indent=2)
        print(f"[VLMGate] Log saved: {self.log_path}  "
              f"({self.total_vlm_calls} calls / {self.total_steps} steps)")
'''

with open('/kaggle/working/vlm_gate.py', 'w') as f:
    f.write(VLM_GATE_SRC)
print("Written: /kaggle/working/vlm_gate.py")

Written: /kaggle/working/vlm_gate.py


## Cell 7: Write VLM Gym Wrapper

This wraps the raw Atari env so `env.step()` passes through the VLM gate.
The shaped reward `= env_reward + vlm_bonus` is what DreamerV3 trains on.

In [9]:
VLM_GYM_WRAPPER_SRC = r'''
import sys
import numpy as np

sys.path.insert(0, "/kaggle/working")
from vlm_gate import VLMGate


class VLMGymWrapper:
    """Wraps a raw gym Atari env with a VLMGate.

    env.step() returns shaped_reward = env_reward + vlm_reward_bonus
    so DreamerV3 trains on VLM-shaped rewards during override windows.

    IMPORTANT: We do NOT clip rewards. DreamerV3 uses symlog internally
    to handle reward magnitudes across domains. Clipping destroys signal.
    """

    def __init__(self, env, api_key="", threshold_z=2.5,
                 queue_len=6, cooldown_steps=80, reward_scale=0.25,
                 warmup_steps=5000):
        self.env  = env
        self.gate = VLMGate(
            api_key        = api_key,
            threshold_z    = threshold_z,
            queue_len      = queue_len,
            cooldown_steps = cooldown_steps,
            reward_scale   = reward_scale,
            warmup_steps   = warmup_steps,
            enabled        = True,
            log_path       = "/kaggle/working/vlm_call_log.json",
        )
        self._step_count            = 0
        self._score                 = 0.0
        self._lives                 = 0
        self._action_hist           = []
        self._current_obs           = None
        self.total_vlm_reward_added = 0.0
        self.total_reward_steps     = 0

        # Proxy standard gym attributes
        for attr in ("observation_space", "action_space",
                     "reward_range", "metadata", "spec"):
            try:
                setattr(self, attr, getattr(env, attr))
            except AttributeError:
                pass

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        if isinstance(result, tuple) and len(result) == 2:
            obs, info = result
        else:
            obs, info = result, {}
        self._current_obs = np.array(obs)
        self.gate.reset_episode()
        self._step_count  = 0
        self._score       = 0.0
        self._lives       = 0
        self._action_hist = []
        if isinstance(result, tuple) and len(result) == 2:
            return obs, info
        return obs

    def step(self, action):
        frame = (self._current_obs if self._current_obs is not None
                 else np.zeros((210, 160, 3), np.uint8))

        final_action, was_override, reward_bonus = self.gate.get_action(
            frame          = frame,
            agent_action   = int(action),
            step           = self._step_count,
            score          = self._score,
            lives          = self._lives,
            action_history = self._action_hist[-6:],
        )

        result = self.env.step(final_action)
        if len(result) == 5:
            obs, env_reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, env_reward, done, info = result

        # NO reward clipping! DreamerV3 handles magnitude via symlog.
        shaped_reward = float(env_reward) + reward_bonus

        if reward_bonus != 0.0:
            self.total_vlm_reward_added += reward_bonus
            self.total_reward_steps     += 1

        self._current_obs  = np.array(obs)
        self._score       += float(env_reward)
        self._step_count  += 1
        self._action_hist.append(int(final_action))
        if isinstance(info, dict) and "lives" in info:
            self._lives = int(info["lives"])

        return obs, shaped_reward, done, info

    def __getattr__(self, name):
        return getattr(self.env, name)

    def close(self):
        self.gate.save_log()
        steps = self.total_reward_steps
        total = self.total_vlm_reward_added
        if steps > 0:
            print(f"[VLMGymWrapper] Reward shaping: {steps} steps shaped | "
                  f"total bonus={total:+.2f} | avg/step={total/steps:+.4f}")
        else:
            print("[VLMGymWrapper] No reward shaping applied.")
        self.env.close()

    def render(self, *a, **kw):
        return self.env.render(*a, **kw)

    def seed(self, seed=None):
        if hasattr(self.env, "seed"):
            return self.env.seed(seed)
'''

with open('/kaggle/working/vlm_gym_wrapper.py', 'w') as f:
    f.write(VLM_GYM_WRAPPER_SRC)
print("Written: /kaggle/working/vlm_gym_wrapper.py")

Written: /kaggle/working/vlm_gym_wrapper.py


## Cell 8: Patch DreamerV3's atari.py to Inject VLM Wrapper

This injects a monkey-patch at the top of `atari.py` so that when the
`VLM_GATE_ENABLED` env variable is set, every `gym.make()` call automatically
wraps the environment with our VLMGymWrapper.

In [10]:
import os, glob

candidates = glob.glob('/kaggle/working/dreamerv3/**/atari.py', recursive=True)
atari_path = None
for c in candidates:
    if 'embodied' in c and 'envs' in c:
        atari_path = c
        break
if atari_path is None and candidates:
    atari_path = candidates[0]

if atari_path is None:
    print('ERROR: atari.py not found -- run Cell 3 (clone) first.')
else:
    print(f'Found: {atari_path}')
    with open(atari_path, 'r') as f:
        original = f.read()

    marker = 'VLM Gate injection'
    if marker in original or 'VLMGymWrapper' in original:
        print('Already patched -- nothing to do.')
    else:
        INJECT = '''
# -- VLM Gate injection (Deliverable 4 - Boxing) ---------------------------------
import os as _d4os, sys as _d4sys

if _d4os.environ.get('VLM_GATE_ENABLED'):
    _d4sys.path.insert(0, '/kaggle/working')
    try:
        from vlm_gym_wrapper import VLMGymWrapper as _VLMW

        def _d4wrap_env(_env):
            return _VLMW(
                _env,
                api_key        = _d4os.environ.get('GEMINI_API_KEY', ''),
                threshold_z    = float(_d4os.environ.get('VLM_THRESHOLD',    '2.5')),
                queue_len      = int(_d4os.environ.get('VLM_QUEUE_LEN',      '6')),
                cooldown_steps = int(_d4os.environ.get('VLM_COOLDOWN',       '80')),
                reward_scale   = float(_d4os.environ.get('VLM_REWARD_SCALE', '0.25')),
                warmup_steps   = int(_d4os.environ.get('VLM_WARMUP_STEPS',   '5000')),
            )

        _d4patched = False
        for _modname in ('gymnasium', 'gym'):
            try:
                _d4mod = __import__(_modname)
            except Exception:
                continue
            if hasattr(_d4mod, 'make'):
                _d4orig_make = _d4mod.make
                def _d4make(*args, __orig=_d4orig_make, **kwargs):
                    return _d4wrap_env(__orig(*args, **kwargs))
                _d4mod.make = _d4make
                _d4patched = True
                globals()['make'] = _d4make

        if _d4patched:
            print('[DreamerV3/Atari] VLM gate + reward shaping active!')
        else:
            print('[DreamerV3/Atari] VLM gate: could not find make() to patch.')
    except Exception as _vlme:
        print(f'[DreamerV3/Atari] VLM gate injection failed: {_vlme}')
# ---------------------------------------------------------------------------------

'''
        patched = INJECT + "\n" + original
        with open(atari_path, 'w') as f:
            f.write(patched)
        print('atari.py patched successfully.')

Found: /kaggle/working/dreamerv3/embodied/envs/atari.py
atari.py patched successfully.


## Cell 9: Run DreamerV3 BASELINE on Boxing (no VLM)

First we run a pure baseline so we have a clean comparison.

In [ ]:
import os, sys, subprocess, shutil

LOGDIR_BASE = '/kaggle/working/dreamer_boxing_baseline'
os.makedirs(LOGDIR_BASE, exist_ok=True)

env = os.environ.copy()
env.pop('XLA_FLAGS', None)
env.pop('TF_XLA_FLAGS', None)
env['JAX_PLATFORMS']     = 'cuda'
env['PYTHONNOUSERSITE']  = '1'

# VLM gate DISABLED for baseline
env.pop('VLM_GATE_ENABLED', None)

cmd = [
    sys.executable, '-u', 'dreamerv3/main.py',
    '--logdir', LOGDIR_BASE,
    '--configs', 'atari100k', 'size12m',
    '--task', 'atari_boxing',
    '--run.steps', '400000',
]

print('='*60)
print('BASELINE RUN: DreamerV3 on Boxing (no VLM)')
print(f'Task:   atari_boxing')
print(f'Config: atari100k + size12m')
print(f'Steps:  400,000')
print(f'Logdir: {LOGDIR_BASE}')
print('='*60)
print()

subprocess.run(cmd, env=env, check=True)

print('\nBaseline training complete!')
zip_path = '/kaggle/working/backup_boxing_baseline'
shutil.make_archive(zip_path, 'zip', LOGDIR_BASE)
print(f'Backup: {zip_path}.zip ({os.path.getsize(zip_path + ".zip")/1024/1024:.1f} MB)')

BASELINE RUN: DreamerV3 on Boxing (no VLM)
Task:   atari_boxing
Config: atari100k + size12m
Steps:  400,000
Logdir: /kaggle/working/dreamer_boxing_baseline

---  ___                           __   ______ ---
--- |   \ _ _ ___ __ _ _ __  ___ _ \ \ / /__ / ---
--- | |) | '_/ -_) _` | '  \/ -_) '/\ V / |_ \ ---
--- |___/|_| \___\__,_|_|_|_\___|_|  \_/ |___/ ---
Replica: 0 / 1
Logdir: /kaggle/working/dreamer_boxing_baseline
Run script: train


A.L.E: Arcade Learning Environment (version 0.9.0+750d7f9)
[Powered by Stella]


Observations
  image            Space(uint8, shape=(96, 96, 1), low=0, high=255)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
Actions
  action           Space(int32, shape=(), low=0, high=18)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(2048,), low=-inf, high=inf)
  dyn/stoch        Space(float32, shape=(32, 16), low=-inf, high=inf)
JAX devices (2): [cuda:0, cuda:1]
Policy devices: cuda:0
Train devices:  cuda:0
Initializing parameters...
Optimizer opt has 11,808,850 params:
     6,310,400 dyn
     2,256,001 dec
       853,503 val
       792,594 pol
       721,407 rew
       656,129 con
       218,816 enc
Done initializing!
Compiling 1 checkpoi

2026-05-06 08:13:07.702472: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778055187.922319     182 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778055187.985997     182 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778055188.511998     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778055188.512086     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778055188.512095     182 computation_placer.cc:177] computation placer alr


--------------------[Agent Step 6_080]--------------------
Metrics filtered by: 'score|length|fps|ratio|train/loss/|train/rand/'
train/loss/con 0.36 / train/loss/dyn 8.73 / train/loss/image 232.35 / train/loss/policy -4.9e-5 / train/loss/rep 8.73 / train/loss/repval 11.02 / train/loss/rew 5.51 / train/loss/value 2.15 / train/rand/action 1 / replay/replay_ratio 78.57 / fps/policy 12.24 / fps/train 899.05

Writing metrics: /kaggle/working/dreamer_boxing_baseline/scores.jsonlWriting metrics: /kaggle/working/dreamer_boxing_baseline/metrics.jsonl

Stop JAX profiler

--------------------[Agent Step 8_080]--------------------
Metrics filtered by: 'score|length|fps|ratio|train/loss/|train/rand/'
episode/score 0 / episode/length 1779 / train/loss/con 0.04 / train/loss/dyn 8.28 / train/loss/image 71.13 / train/loss/policy 6.4e-4 / train/loss/rep 8.28 / train/loss/repval 10.35 / train/loss/rew 5.11 / train/loss/value 8.33 / train/rand/action 1 / replay/replay_ratio 260 / fps/policy 4.15 / fps/tr

## Cell 10: Run DreamerV3 + VLM on Boxing

Now we run with the VLM surprise gate active.

In [1]:
import os, sys, subprocess, shutil

LOGDIR_VLM = '/kaggle/working/dreamer_boxing_vlm'
os.makedirs(LOGDIR_VLM, exist_ok=True)

env = os.environ.copy()
env.pop('XLA_FLAGS', None)
env.pop('TF_XLA_FLAGS', None)
env['JAX_PLATFORMS']     = 'cuda'
env['PYTHONNOUSERSITE']  = '1'

# VLM gate ENABLED
env['VLM_GATE_ENABLED']  = '1'
env['GEMINI_API_KEY']    = GEMINI_API_KEY
env['VLM_THRESHOLD']     = '2.5'
env['VLM_QUEUE_LEN']     = '6'
env['VLM_COOLDOWN']      = '80'
env['VLM_REWARD_SCALE']  = '0.25'
env['VLM_WARMUP_STEPS']  = '5000'

cmd = [
    sys.executable, '-u', 'dreamerv3/main.py',
    '--logdir', LOGDIR_VLM,
    '--configs', 'atari100k', 'size12m',
    '--task', 'atari_boxing',
    '--run.steps', '400000',
]

print('='*60)
print('VLM-GATED RUN: DreamerV3 + Gemini on Boxing')
print(f'Task:      atari_boxing')
print(f'Config:    atari100k + size12m')
print(f'Steps:     400,000')
print(f'Logdir:    {LOGDIR_VLM}')
print(f'Threshold: z=2.5  queue=6  cooldown=80  warmup=5000')
print('='*60)
print()
print('VLM calls will appear as: [VLMGate] step=XXXXX ...')
print()

subprocess.run(cmd, env=env, check=True)

print('\nVLM-gated training complete!')
zip_path = '/kaggle/working/backup_boxing_vlm'
shutil.make_archive(zip_path, 'zip', LOGDIR_VLM)
print(f'Backup: {zip_path}.zip ({os.path.getsize(zip_path + ".zip")/1024/1024:.1f} MB)')

NameError: name 'GEMINI_API_KEY' is not defined

---
## Cell 11: Load Metrics from Both Runs

In [ ]:
import json, glob, os
import numpy as np

LOGDIR_BASE = "/kaggle/working/dreamer_boxing_baseline"
LOGDIR_VLM  = "/kaggle/working/dreamer_boxing_vlm"

def load_metrics(logdir):
    """Load all records from metrics.jsonl files in logdir."""
    files = glob.glob(os.path.join(logdir, "**", "metrics.jsonl"), recursive=True)
    if not files:
        files = glob.glob(os.path.join(logdir, "**", "*.jsonl"), recursive=True)
    if not files:
        return []
    records = []
    for fpath in files:
        with open(fpath) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
    return records

def extract(records, candidates):
    """Extract (steps, values) arrays for the first matching key."""
    steps, vals = [], []
    for r in records:
        step = r.get("step", r.get("steps"))
        if step is None:
            continue
        for k in candidates:
            if k in r and isinstance(r[k], (int, float)):
                steps.append(step)
                vals.append(r[k])
                break
    return np.array(steps, dtype=np.float64), np.array(vals, dtype=np.float64)

# Keys to search for in JSONL (DreamerV3 may use different names)
SCORE_KEYS = ["episode/score", "train/episode_score", "episode_score",
              "return", "reward", "episode/return"]
DYN_KEYS   = ["train/loss/dyn", "dyn_loss", "loss/dyn"]

base_records = load_metrics(LOGDIR_BASE)
vlm_records  = load_metrics(LOGDIR_VLM)

base_s_steps, base_scores = extract(base_records, SCORE_KEYS)
vlm_s_steps,  vlm_scores  = extract(vlm_records,  SCORE_KEYS)
base_d_steps, base_dyn    = extract(base_records, DYN_KEYS)
vlm_d_steps,  vlm_dyn     = extract(vlm_records,  DYN_KEYS)

print(f"Baseline  : {len(base_scores):>5d} score entries, {len(base_dyn):>5d} dyn-loss entries")
print(f"VLM-gated : {len(vlm_scores):>5d} score entries, {len(vlm_dyn):>5d} dyn-loss entries")

if len(base_scores) > 0:
    print(f"  baseline  max={base_scores.max():.1f}  mean={base_scores.mean():.1f}  final={base_scores[-1]:.1f}")
if len(vlm_scores) > 0:
    print(f"  vlm-gated max={vlm_scores.max():.1f}  mean={vlm_scores.mean():.1f}  final={vlm_scores[-1]:.1f}")

## Cell 12: Load VLM Call Log

In [ ]:
import json, os

VLM_LOG_PATH = '/kaggle/working/vlm_call_log.json'

vlm_log = {'total_steps': 0, 'total_vlm_calls': 0, 'call_pct': 0.0, 'calls': []}
if os.path.exists(VLM_LOG_PATH):
    with open(VLM_LOG_PATH) as f:
        vlm_log = json.load(f)
    print(f"VLM call log loaded: {VLM_LOG_PATH}")
    print(f"  Total env steps  : {vlm_log['total_steps']:,}")
    print(f"  Total VLM calls  : {vlm_log['total_vlm_calls']:,}")
    print(f"  Call rate        : {vlm_log['call_pct']:.3f}%")
    calls = vlm_log.get('calls', [])
    if calls:
        print(f"  First call step  : {calls[0]['step']}")
        print(f"  Last  call step  : {calls[-1]['step']}")
        bonuses = [c.get('reward_bonus', 0) for c in calls]
        print(f"  Avg reward bonus : {sum(bonuses)/len(bonuses):+.3f}")
        print(f"  Max reward bonus : {max(bonuses):+.3f}")
        if 'description' in calls[0]:
            print(f"  Example VLM desc : {calls[0]['description'][:120]}")
else:
    print(f'No VLM call log found at {VLM_LOG_PATH}')
    print('This is expected if training has not completed yet.')

## Cell 13: Plot 1 — Score Comparison (Baseline vs VLM-Gated)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

def smooth(arr, window=10):
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='valid')

fig, ax = plt.subplots(figsize=(10, 5))

if len(base_scores) > 0:
    ax.plot(base_s_steps[:len(smooth(base_scores))], smooth(base_scores),
            label='Baseline (DreamerV3)', color='royalblue', alpha=0.85)
if len(vlm_scores) > 0:
    ax.plot(vlm_s_steps[:len(smooth(vlm_scores))], smooth(vlm_scores),
            label='VLM-Gated (DreamerV3 + Gemini)', color='darkorange', alpha=0.85)

# Reference lines from paper
ax.axhline(y=82, color='green', linestyle='--', alpha=0.5, label='Paper Dreamer (400K): 82')
ax.axhline(y=12, color='gray', linestyle=':', alpha=0.5, label='Human gamer: 12')
ax.axhline(y=0,  color='lightgray', linestyle=':', alpha=0.3, label='Random: 0')

ax.set_xlabel('Environment Steps')
ax.set_ylabel('Episode Score')
ax.set_title('Boxing — DreamerV3 Baseline vs VLM-Gated')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/plot1_scores.png', dpi=150)
print('saved: plot1_scores.png')
plt.show()

## Cell 14: Plot 2 — Dynamics Loss Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

if len(base_dyn) > 0:
    ax.plot(base_d_steps[:len(smooth(base_dyn, 20))], smooth(base_dyn, 20),
            label='Baseline dyn loss', color='royalblue', alpha=0.7)
if len(vlm_dyn) > 0:
    ax.plot(vlm_d_steps[:len(smooth(vlm_dyn, 20))], smooth(vlm_dyn, 20),
            label='VLM-gated dyn loss', color='darkorange', alpha=0.7)

ax.set_xlabel('Environment Steps')
ax.set_ylabel('Dynamics Loss')
ax.set_title('Boxing — World Model Dynamics Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/plot2_dynloss.png', dpi=150)
print('saved: plot2_dynloss.png')
plt.show()

## Cell 15: Plot 3 — VLM Call Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

calls = vlm_log.get('calls', [])
if calls:
    call_steps   = [c['step'] for c in calls]
    call_bonuses = [c.get('reward_bonus', 0) for c in calls]

    colors = ['green' if b >= 0 else 'red' for b in call_bonuses]
    ax.scatter(call_steps, call_bonuses, c=colors, s=30, alpha=0.7, edgecolors='k', linewidths=0.3)
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Environment Step')
    ax.set_ylabel('Reward Bonus')
    ax.set_title(f'VLM Calls Timeline ({len(calls)} total calls, '
                 f'{vlm_log.get("call_pct", 0):.3f}% of steps)')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No VLM calls recorded', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='gray')
    ax.set_title('VLM Calls Timeline (no calls)')

plt.tight_layout()
plt.savefig('/kaggle/working/plot3_vlm_calls.png', dpi=150)
print('saved: plot3_vlm_calls.png')
plt.show()

## Cell 16: Plot 4 — Score vs Dynamics Loss (Dual Axis)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Use VLM-gated run if available, else baseline
if len(vlm_scores) > 0:
    s_steps, s_vals = vlm_s_steps, vlm_scores
    d_steps, d_vals = vlm_d_steps, vlm_dyn
    label_prefix = 'VLM-gated'
else:
    s_steps, s_vals = base_s_steps, base_scores
    d_steps, d_vals = base_d_steps, base_dyn
    label_prefix = 'Baseline'

if len(s_vals) > 0:
    ax1.plot(s_steps[:len(smooth(s_vals))], smooth(s_vals),
             color='royalblue', label=f'{label_prefix} Score')
    ax1.set_ylabel('Episode Score', color='royalblue')

if len(d_vals) > 0:
    ax2.plot(d_steps[:len(smooth(d_vals, 20))], smooth(d_vals, 20),
             color='coral', alpha=0.6, label=f'{label_prefix} Dyn Loss')
    ax2.set_ylabel('Dynamics Loss', color='coral')

# Mark VLM calls
calls = vlm_log.get('calls', [])
if calls:
    for c in calls:
        ax1.axvline(x=c['step'], color='green', alpha=0.15, linewidth=0.5)

ax1.set_xlabel('Environment Steps')
ax1.set_title(f'Boxing — Score & Dynamics Loss ({label_prefix})')
ax1.grid(True, alpha=0.2)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.savefig('/kaggle/working/plot4_score_vs_loss.png', dpi=150)
print('saved: plot4_score_vs_loss.png')
plt.show()

## Cell 17: Final Summary

In [ ]:
import os, glob
import numpy as np

print("=" * 65)
print("FINAL RESULTS SUMMARY — Boxing (DreamerV3 + VLM Gate)")
print("=" * 65)
print()
print("Model         : DreamerV3 (Hafner et al., 2023/2025)")
print("Config        : atari100k + size12m")
print("Environment   : Atari Boxing (ALE/Boxing-v5)")
print("VLM           : Gemini 1.5 Flash (free tier)")
print("Steps         : 400,000 env steps")
print()

def summarise(label, scores, steps):
    if len(scores) == 0:
        print(f"{label:<14}: no data")
        return
    print(f"{label:<14}: max={scores.max():.0f}  mean={scores.mean():.1f}  "
          f"final={scores[-1]:.0f}  episodes={len(scores)}")

summarise("Baseline",   base_scores, base_s_steps)
summarise("VLM-gated",  vlm_scores,  vlm_s_steps)

print()
print("Paper reference scores (Boxing):")
print("  Random:   0")
print("  Human:   12")
print("  DreamerV3 (400K, Atari100k): ~82")
print("  DreamerV3 (200M):           100")

print()
calls = vlm_log.get('calls', [])
total = vlm_log.get('total_steps', 0)
print(f"VLM calls       : {len(calls)}")
print(f"Steps covered   : {total:,}")
print(f"Call rate        : {vlm_log.get('call_pct', 0):.3f}%")
if total > 0:
    print(f"Calls saved vs always-on: {100 - vlm_log.get('call_pct', 0):.1f}%")
if calls:
    bonuses = [c.get('reward_bonus', 0) for c in calls]
    print(f"Avg reward bonus: {sum(bonuses)/len(bonuses):+.3f}")

print()
print("Saved files:")
for p in sorted(glob.glob('/kaggle/working/plot*.png')):
    sz = os.path.getsize(p) / 1024
    print(f"  {p}  ({sz:.0f} KB)")
for p in sorted(glob.glob('/kaggle/working/backup_*.zip')):
    sz = os.path.getsize(p) / 1024 / 1024
    print(f"  {p}  ({sz:.1f} MB)")
if os.path.exists('/kaggle/working/vlm_call_log.json'):
    sz = os.path.getsize('/kaggle/working/vlm_call_log.json') / 1024
    print(f"  /kaggle/working/vlm_call_log.json  ({sz:.1f} KB)")
print("=" * 65)